In [3]:
import pandas as pd

from string import punctuation

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import spacy
from spacy.matcher import Matcher

import json

In [7]:
concept_embeddings = {}
TF_concept_embeddings = {}

drinks = pd.read_json(r"Data/drinks.json")
drink_names = {drinks["id"][i]: drinks["aliases"][i] for i in range(len(drinks["id"]))}

concept_embeddings_model = SentenceTransformer("all-MiniLM-L6-v2")
TF_concept_embeddings_model = SentenceTransformer("all-MiniLM-L6-v2")


with open("Data/attribute_parse.json", "r", encoding="utf-8") as f:
    ATTRIBUTE_CONCEPTS = json.load(f)
    
for concept, words in ATTRIBUTE_CONCEPTS.items():
    concept_embeddings[concept] = concept_embeddings_model.encode(words)


with open("Data/TF_attribute_parse.json", "r", encoding="utf-8") as f:
    TF_ATTRIBUTE_CONCEPTS = json.load(f)
    
for concept, words in TF_ATTRIBUTE_CONCEPTS.items():
    TF_concept_embeddings[concept] = TF_concept_embeddings_model.encode(words)

In [14]:
class Sentence:

    def __init__(self, sentence, nlp):

        self.sentence = sentence
        self.nlp = nlp 



    def clean(self):

        sentence = self.sentence.lower()
        nlp = self.nlp

        #turn words into their main form
        doc = nlp(sentence)
        words = []

        for word in doc:
            if ' ' not in str(word): #check for extra spaces
                words.append(word.lemma_)
        
        ' '.join(words)

        translator = str.maketrans('', '', punctuation)
        cleaned_sentence = sentence.translate(translator)


        self.tokens = cleaned_sentence.split()

        return cleaned_sentence.split()



    def change_drink_to_id(self):

        tokens = self.tokens

        nlp = self.nlp
        doc = nlp(" ".join(tokens))

        references = []
        reference_ids = []

        relatives = [] #which word is each token ralated to
        
        border = 0

        for i in range(len(tokens)):

            for drink_name, drink_val in drink_names.items():

                if i != 0 and i == border:
                    break

                #check if sny drink name is mentioned
                if tokens[i] in drink_val: 

                    #check if the drink is iced, to replace "iced" and the drink name with a single word
                    if i > 0 and tokens[i-1] == "iced":
                        tokens[i] = f"iced_{drink_name}"

                        references.append(tokens[i])
                        reference_ids.append(i-1)

                        tokens.pop(i-1)
                        tokens.append('')
                        border-=1 #border avoids empty indexses, since an elemet has been removed

                    else:
                        tokens[i] = drink_name
                        references.append(tokens[i])
                        reference_ids.append(i)
                    

        if border:
            tokens = tokens[0:border]

        for token_i in range(len(doc)):
            relatives.append(doc[token_i].head.i)

        
        self.relatives = relatives
        self.tokens = tokens
        self.references = reference_ids

        return tokens, reference_ids



    def find_ADJ(self):

        tokens = self.tokens
        nlp = self.nlp
        doc = nlp(" ".join(tokens))

        similarity_tokens_strt = self.similarity_tokens[0]

        adjs = []
        adjs_index = []
        comp_more = []
        comp_less = []
        superlatives = []

        doc = nlp(" ".join(tokens))


        for token_i in range(len(tokens)):
            
            if self.find_concept(tokens[token_i]) and token_i not in similarity_tokens_strt and token_i:
                
                adjs.append(doc[token_i].text)
                adjs_index.append(token_i)

                word = tokens[token_i]
                
                if tokens[token_i - 1] == "less":
                    comp_less.append(token_i)

                elif word[-2:] == "er" and nlp(word[:-2])[0].pos_ == "ADJ" or tokens[token_i - 1] == "more":
                    comp_more.append(token_i)

                elif tokens[token_i][-3:] == "est" and nlp(word[:-3])[0].pos_ == "ADJ" or tokens[token_i - 1] == "most":
                    superlatives.append(token_i)


        self.adjs_index = adjs_index
        self.comp_more = comp_more
        self.comp_less = comp_less
        self.adjectives = adjs

        return (adjs, adjs_index, comp_more, comp_less, superlatives)



    #find the reference of adjectives
    def ADJ_owner(self): 

        tokens = self.tokens
        adj_indexes = self.adjs_index
        nlp = self.nlp
        comp_more = self.comp_more

        similarity_tokens = self.similarity_tokens
        similarity_owners = self.similarity_owners

        doc = nlp(" ".join(tokens))

        owners = []
        relatives = self.relatives


        # the function searches through the main relative of the given word, until it reaches either the root of the sentence or a drink name
        def find_rel(indexses):

            for i in indexses:

                #try to find i as a relative to a token. If failed, set it as None
                try:
                    word_rel2adj = relatives.index(i)
                except ValueError:
                    word_rel2adj = None

                try:
                    if i in comp_more and tokens[i + 1] == "than" and tokens[i + 2] in drink_names:
                        return i+2
                except IndexError:
                    pass


                if tokens[ relatives[i] ] in drink_names:
                    return relatives[i]

                elif word_rel2adj is not None and tokens[word_rel2adj] in drink_names:
                    return word_rel2adj

                #check if the word related to an adjective is inside of the similarity_ends list
                elif word_rel2adj in similarity_tokens[0]:
                    #return the owners of the similarity-phrase
                    return similarity_owners[ similarity_tokens[0].index( relatives.index(i) ) ]

                #check if the word related to an adjective is inside of the similarity_starts list
                elif word_rel2adj in similarity_tokens[1]:
                    return similarity_owners[ similarity_tokens[1].index( relatives.index(i) ) ]
                
                #if our adjective word is related to another adjective, then the head word might be the relative of the relative
                elif i in adj_indexes and relatives[i] in adj_indexes:

                    if  relatives[i] != i: #avoiding infinite loops if the relative of the adjective was in any case itself
                        indexses.append(relatives[i])
                        indexses.pop(0)
        
                        return find_rel(indexses)

                    else:
                        return False
    
                elif tokens[i] in drink_names:
                    return i

                #if the relative was the root, it means no suitable reference(owner) was found. So return False
                elif doc[ relatives[i] ].dep_ == "ROOT":
                    return False
                

                temp_list = [j for j in range(len(relatives)) if relatives[j] == i or relatives[i] == j]
                indexses = indexses + temp_list
                indexses.pop(0)

                return find_rel(indexses)
            

        for adj_i in adj_indexes:
            owners.append(find_rel([adj_i]))


        self.relatives = relatives
        self.adj_owners = owners

        return owners


    #return the main form of an adjective (if the adjective was inside if the list of our drink attributes)
    def find_concept(self, word):
                
            best_score = 0
            best_concept = None

            word_vector = concept_embeddings_model.encode(word)


            for concept, vectors in concept_embeddings.items():
                scores = cosine_similarity(
                    [word_vector],
                    vectors
                )[0]

                score = max(scores)

                if score > best_score:
                    best_score = score
                    best_concept = concept


            #add a False value to the list if the adjective was irrelevant(it had a low score)
            if best_score > 0.6:
                return best_concept

            else:
                return False



    def reform_adjs(self):

        tokens = self.tokens

        reformed_ajds = []


        for i in range(len(tokens)):

            result = self.find_concept(tokens[i])

            if result:
                reformed_ajds.append(result)
                

        self.reformed_ajds = reformed_ajds


        return reformed_ajds
    


    #return the main form of an attribute, which is set as either True or False (if the adjective was inside if the list of our drink attributes)
    def find_TF_concept(self, word):
        best_score = 0
        best_concept = None

        word_vector = TF_concept_embeddings_model.encode(word)


        for concept, vectors in TF_concept_embeddings.items():
            scores = cosine_similarity(
                [word_vector],
                vectors
            )[0]

            score = max(scores)

            if score > best_score:
                best_score = score
                best_concept = concept


        #ad a False value to the list if the adjective was irrelevant(it had a low score)
        if best_score > 0.6:
            return best_concept

        else:
            return False


    
    def reform_TF_adjs(self):
    
            tokens = self.tokens
    
            reformed_TF_ajds = []
            tf_adjs_index = []
    
    
            for i in range(len(tokens)):
    
                result = self.find_TF_concept(tokens[i])
    
                if result:
                    
                    reformed_TF_ajds.append(result)
                    tf_adjs_index.append(i)
                    
    
            self.reformed_TF_ajds = reformed_TF_ajds
            self.tf_adjs_index = tf_adjs_index
    
    
            return reformed_TF_ajds



    #find words that refer an adjective to a drink name, such as "like", "similar to"
    def find_similarity(self):

        tokens = self.tokens
        nlp = self.nlp
        doc = nlp(' '.join(self.tokens))

        matcher = Matcher(nlp.vocab)
                
        patterns = [
            [{"LOWER": "like"}],
            [
                {"LOWER": "same"},
                {"LOWER": "as"}
            ],
            [
                {"LOWER": "similar"},
                {"LOWER": "to"}
            ],
            [
                {"LEMMA": "resemble"}
            ],
            [
                {"LOWER": "equivalent"},
                {"LOWER": "to"}
            ],
            [
                {"LOWER": "just", "OP": "?"},
                {"LOWER": "like"}
            ],
            [
                {"LOWER": "as"},
                {},
                {"LOWER": "as"}
            ]
        ]
        
        matcher.add("SIMILARITY", patterns)
        matches = matcher(doc)

        relatives = []
        owners = []

        #a list of tokens, with which the "similarity" phrase ends
        similarity_tokens_end = []
        #a list of tokens, with which the "similarity" phrase starts
        similarity_tokens_strt = []

        for token_i in range(len(doc)):
            relatives.append(doc[token_i].head.i)
            

        for i, start, end in matches:
            if doc[end-1].pos_ != "VERB":

                similarity_tokens_end.append(end-1)
                similarity_tokens_strt.append(start)

                try:
                        if tokens[end] in drink_names:
                            owners.append(end)

                        elif relatives[end] in drink_names:
                            owners.append(relatives[end])

                        else:
                            for reference in self.references:
                                if reference > end-1:
                                    owners.append(reference)

                except:
                    pass


        self.similarity_tokens = (similarity_tokens_strt, similarity_tokens_end)
        self.similarity_owners = owners
        
        return similarity_tokens_end, similarity_tokens_strt, owners

In [15]:
nlp = spacy.load("en_core_web_sm")

Sentence1 = Sentence("I would like something like a sweet latte", nlp)

Sentence1.clean()
Sentence1.change_drink_to_id()

Sentence1.find_similarity()

Sentence1.find_ADJ()
Sentence1.reform_TF_adjs()
Sentence1.ADJ_owner()

[7]